# E1: compare scores with and without speaker weights

Use the same saved validation predictions for each classifier's 39-feature baseline and selected count.

- **Without weights:** each clip has equal influence within its class.
- **With weights:** each speaker has equal influence within its class.

Keep models, predictions, and feature counts fixed. This comparison changes the scoring weights only.


## 1. Load validation results

Run the final training section of `E1_acoustic_feature_count_comparison.ipynb` first.


In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import balanced_accuracy_score

results_file = Path.cwd() / 'outputs/e1_feature_study/results/study_results.joblib'
if not results_file.is_file():
    raise FileNotFoundError('Run the main study through final training first.')

results = joblib.load(results_file)
predictions = pd.DataFrame(results['validation_predictions'])
selected_counts = results['selected_counts']
print('Results:', results_file)


## 2. Check the comparison data

All models must use the same validation clips, labels, and speaker IDs.


In [ ]:
if not selected_counts:
    raise ValueError('No selected feature counts were saved.')
if set(predictions['split']) != {'validation'}:
    raise ValueError('Use validation predictions only.')

clip_metadata = predictions[['clip_id', 'speaker_group', 'true_label']].drop_duplicates()
if clip_metadata['clip_id'].duplicated().any():
    raise ValueError('Models have different labels or speaker IDs for the same clip.')
if set(clip_metadata['true_label']) != {0, 1}:
    raise ValueError('Both classes must occur in validation data.')
if (clip_metadata.groupby('speaker_group')['true_label'].nunique() != 1).any():
    raise ValueError('Each speaker must belong to one class.')

expected_clip_ids = set(clip_metadata['clip_id'])
for model, selected_count in selected_counts.items():
    if not 1 <= selected_count <= 39:
        raise ValueError(f'Invalid feature count for {model}: {selected_count}')
    for count in {39, selected_count}:
        rows = predictions[(predictions['model'] == model) & (predictions['feature_count'] == count)]
        if rows['clip_id'].duplicated().any() or set(rows['clip_id']) != expected_clip_ids:
            raise ValueError(f'Missing or duplicate validation clips: {model}, {count} features')


## 3. Calculate both scores

Each clip's speaker weight is `1 / number of clips from its speaker`.
A speaker with ten clips gives each clip weight 0.1. The speaker's total weight is 1.
If the selected count is 39, reuse the baseline scores.


In [ ]:
def calculate_scores(rows):
    truth = rows['true_label']
    predicted = rows['prediction']
    clips_per_speaker = rows.groupby('speaker_group')['clip_id'].transform('size')
    speaker_weights = 1.0 / clips_per_speaker

    weighted = balanced_accuracy_score(truth, predicted, sample_weight=speaker_weights)
    unweighted = balanced_accuracy_score(truth, predicted)
    return {
        'speaker_weighted_accuracy_pct': 100 * weighted,
        'ordinary_balanced_accuracy_pct': 100 * unweighted,
        'unweighted_minus_weighted_points': 100 * (unweighted - weighted),
    }


comparison_rows = []
for model, selected_count in selected_counts.items():
    model_predictions = predictions[predictions['model'] == model]
    baseline_rows = model_predictions[model_predictions['feature_count'] == 39]
    baseline_scores = calculate_scores(baseline_rows)

    for role, count in [('39-feature baseline', 39), ('Selected count', selected_count)]:
        if count == 39:
            scores = baseline_scores
        else:
            selected_rows = model_predictions[model_predictions['feature_count'] == count]
            scores = calculate_scores(selected_rows)

        comparison_rows.append({
            'model': model, 'role': role, 'feature_count': count, **scores,
            'weighted_gain_from_39_points': (
                scores['speaker_weighted_accuracy_pct'] - baseline_scores['speaker_weighted_accuracy_pct']),
            'unweighted_gain_from_39_points': (
                scores['ordinary_balanced_accuracy_pct'] - baseline_scores['ordinary_balanced_accuracy_pct']),
        })

comparison = pd.DataFrame(comparison_rows)
display(comparison.round(3))


## 4. Validation speaker statistics

Count each validation clip once, even when it appears in several model comparisons.
Show the speaker summary and the distribution of clips per speaker.
Speakers with multiple clips have more influence when speaker weights are removed.


In [ ]:
# clip_metadata contains one row per validation clip.
class_names = {0: 'Non-Singapore', 1: 'Singapore'}
speaker_counts = clip_metadata.groupby(['true_label', 'speaker_group']).size().rename('clips').reset_index()

summary_rows = []
for label, name in [*class_names.items(), (None, 'Total')]:
    counts = speaker_counts['clips'] if label is None else speaker_counts.loc[
        speaker_counts['true_label'] == label, 'clips']
    summary_rows.append({
        'Class': name,
        'Clips': int(counts.sum()),
        'Speakers': len(counts),
        'Speakers with one clip': int((counts == 1).sum()),
        'Speakers with multiple clips': int((counts > 1).sum()),
        'Average clips per speaker': counts.mean(),
        'Clips from repeated speakers': int(counts[counts > 1].sum()),
    })

speaker_summary = pd.DataFrame(summary_rows).set_index('Class')
print('Validation speaker summary')
display(speaker_summary.round(2))

clip_distribution = pd.crosstab(speaker_counts['clips'], speaker_counts['true_label'])
clip_distribution = clip_distribution.rename(columns=class_names)
clip_distribution['Total'] = clip_distribution.sum(axis=1)
clip_distribution.index.name = 'Clips per speaker'
clip_distribution.columns.name = None
print('Distribution of clips per speaker')
display(clip_distribution)

totals = speaker_summary.loc['Total']
repeated_speaker_pct = 100 * totals['Speakers with multiple clips'] / totals['Speakers']
repeated_clip_pct = 100 * totals['Clips from repeated speakers'] / totals['Clips']
print(f"Speakers with multiple clips: {int(totals['Speakers with multiple clips'])} "
      f"of {int(totals['Speakers'])} ({repeated_speaker_pct:.1f}%)")
print(f"Clips from repeated speakers: {int(totals['Clips from repeated speakers'])} "
      f"of {int(totals['Clips'])} ({repeated_clip_pct:.1f}%)")


## 5. Compare the scores

Labels show the percentages on each bar.


In [ ]:
fig, axes = plt.subplots(1, len(selected_counts), figsize=(5 * len(selected_counts), 4),
                         sharey=True, squeeze=False)
bar_width = 0.36

for axis, model in zip(axes[0], selected_counts):
    rows = comparison[comparison['model'] == model]
    positions = np.arange(len(rows))
    weighted_bars = axis.bar(positions - bar_width / 2, rows['speaker_weighted_accuracy_pct'],
                             bar_width, label='With speaker weights')
    unweighted_bars = axis.bar(positions + bar_width / 2, rows['ordinary_balanced_accuracy_pct'],
                               bar_width, label='Without speaker weights')
    axis.bar_label(weighted_bars, fmt='%.2f%%', padding=3, fontsize=8)
    axis.bar_label(unweighted_bars, fmt='%.2f%%', padding=3, fontsize=8)
    labels = [f'{row.role}\n({row.feature_count} features)' for row in rows.itertuples()]
    axis.set_xticks(positions, labels)
    axis.set(title=model.replace('_', ' ').title(), ylim=(0, 100),
             ylabel='Validation balanced accuracy (%)')

axes[0, 0].legend(fontsize=8)
fig.tight_layout()
plt.show()


## Read the results

`unweighted_minus_weighted_points` shows the score change when speaker weights are removed.
A positive value means the score increases without weights.
The two gain columns show whether the selected count improves on 39 features under each metric.

Both metrics give the two classes equal influence.
If all speakers have the same number of clips within each class, both scores are equal.
